# 04 — Training & evaluation across all split schemes — CSE-CIC-IDS2018 — Mutual Information study

Identical protocol to the base study: 5 models × schemes `70_30`, `80_20`, `90_10`,
`70_20_10` (train/val/test) and `cv5` (5-fold stratified CV), on the top 40 features
selected by **Mutual Information**. All splits stratified with the same seed and saved to `splits/`.
Each pipeline scales and SMOTE-balances its own training portion only. A (scheme, model)
whose `metrics.json` already exists is skipped — restarts resume.

In [1]:
DATASET = "cic2018"
FT_ROOT = "ft40_mi"
TARGET_PER_CLASS = 50_000   # per-class row cap for the in-pipeline under/over sampling
MODELS_TO_RUN = None        # None = all 5; or e.g. ["svm", "knn"]
RUN_CV = True

In [2]:
import sys
from pathlib import Path

def _find_nb_common():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for c in (cand, cand / "notebooks"):
            if (c / "nb_common.py").exists():
                return c
    raise FileNotFoundError("nb_common.py not found relative to " + str(Path.cwd()))
sys.path.insert(0, str(_find_nb_common()))

import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import nb_common as nbc
from ids import config
from ids.schema import CANONICAL_COLUMNS
from ids.models.pipelines import build_pipeline
from ids.models.registry import MODEL_NAMES
from ids.evaluation.metrics import evaluate_model
from ids.evaluation.cross_val import cross_validate_models, summarize_cv
from ids.evaluation.plots import plot_confusion_matrix

P = nbc.ft_paths(DATASET, FT_ROOT)
log = nbc.setup_logging(P.logs / "04_experiments.log", "nb04")

df = pd.read_parquet(P.data / "clean.parquet")
X, y = df[CANONICAL_COLUMNS], df["y"].to_numpy()
top40 = nbc.load_json(P.features / "top40.json")["selected"]
mapping = nbc.load_json(P.data / "label_mapping.json")
class_names = [c for c, _ in sorted(mapping.items(), key=lambda kv: kv[1])]
n_classes = len(class_names)
models_to_run = MODELS_TO_RUN or list(MODEL_NAMES)
log.info("%d rows | %d selected features | %d classes | models=%s | target_per_class=%d",
         len(X), len(top40), n_classes, models_to_run, TARGET_PER_CLASS)

2026-07-06 19:05:09,696 INFO nb04: 1586902 rows | 40 selected features | 15 classes | models=['decision_tree', 'random_forest', 'svm', 'naive_bayes', 'knn'] | target_per_class=50000


## 1. Build (or load) all split-scheme indices — stratified, fixed seed, saved to `splits/`

In [3]:
idx = np.arange(len(X))

def holdout(test_size):
    tr, te = train_test_split(idx, test_size=test_size, stratify=y,
                              random_state=config.RANDOM_STATE)
    return {"train_idx": tr, "test_idx": te}

def train_val_test():
    tr_val, te = train_test_split(idx, test_size=0.10, stratify=y,
                                  random_state=config.RANDOM_STATE)
    tr, val = train_test_split(tr_val, test_size=2/9, stratify=y[tr_val],
                               random_state=config.RANDOM_STATE)  # 2/9 of 90% = 20%
    return {"train_idx": tr, "val_idx": val, "test_idx": te}

SPLITS = {}
p_7030 = P.splits / "70_30_indices.npz"
SPLITS["70_30"] = dict(np.load(p_7030)) if p_7030.exists() else holdout(0.30)
SPLITS["80_20"] = holdout(0.20)
SPLITS["90_10"] = holdout(0.10)
SPLITS["70_20_10"] = train_val_test()

for scheme, s in SPLITS.items():
    np.savez_compressed(P.splits / f"{scheme}_indices.npz", **s)
    log.info("%s: %s", scheme,
             " / ".join(f"{k.replace('_idx','')}={len(v)}" for k, v in s.items()))

2026-07-06 19:05:11,607 INFO nb04: 70_30: train=1110831 / test=476071
2026-07-06 19:05:12,412 INFO nb04: 80_20: train=1269521 / test=317381
2026-07-06 19:05:13,207 INFO nb04: 90_10: train=1428211 / test=158691
2026-07-06 19:05:13,999 INFO nb04: 70_20_10: train=1110830 / val=317381 / test=158691


## 2. Train + evaluate every (scheme × model)

In [4]:
def run_one(scheme, name, train_idx, test_idx, val_idx=None):
    out_dir = P.runs / scheme / name
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics.json"
    if metrics_path.exists():
        log.info("SKIP %s/%s — metrics.json exists", scheme, name)
        return nbc.load_json(metrics_path)

    pipeline = build_pipeline(name, top40, target_per_class=TARGET_PER_CLASS)
    t0 = time.perf_counter()
    pipeline.fit(X.iloc[train_idx], y[train_idx])
    fit_seconds = time.perf_counter() - t0
    log.info("%s/%s: fit on %d rows in %.1fs", scheme, name, len(train_idx), fit_seconds)

    record = {"dataset": DATASET, "scheme": scheme, "model": name,
              "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
              "target_per_class": TARGET_PER_CLASS, "fit_seconds": round(fit_seconds, 3)}
    evals = [("test", test_idx, "")]
    if val_idx is not None:
        record["n_val"] = int(len(val_idx))
        evals.append(("validation", val_idx, "_val"))

    for split_name, eidx, suffix in evals:
        m, _, y_pred, y_proba = evaluate_model(name, pipeline, X.iloc[eidx], y[eidx])
        cm = confusion_matrix(y[eidx], y_pred, labels=np.arange(n_classes))
        np.save(out_dir / f"confusion_matrix{suffix}.npy", cm)
        np.save(out_dir / f"predictions{suffix}.npy", y_pred)
        if y_proba is not None:
            np.save(out_dir / f"probas{suffix}.npy", y_proba.astype(np.float32))
        rep = classification_report(y[eidx], y_pred, labels=np.arange(n_classes),
                                    target_names=class_names, output_dict=True,
                                    zero_division=0)
        pd.DataFrame(rep).T.to_csv(out_dir / f"per_class{suffix}.csv")
        if split_name == "test":
            plot_confusion_matrix(f"{name} ({scheme})", cm, class_names, out_dir)
        record[split_name] = m.to_dict()
        log.info("%s/%s [%s]: acc=%.4f f1_macro=%.4f roc_auc=%s det=%.3fms",
                 scheme, name, split_name, m.accuracy, m.f1_macro,
                 "n/a" if m.roc_auc is None else round(m.roc_auc, 4),
                 m.detection_time_ms)

    nbc.save_json(record, metrics_path)
    return record

records = []
with nbc.timed(log, "all holdout schemes"):
    for scheme, s in SPLITS.items():
        with nbc.timed(log, f"scheme {scheme}"):
            for name in models_to_run:
                records.append(run_one(scheme, name, s["train_idx"], s["test_idx"],
                                       s.get("val_idx")))
pd.DataFrame([{ "scheme": r["scheme"], "model": r["model"],
                "fit_s": r["fit_seconds"], **{k: round(v, 4) if isinstance(v, float) else v
                for k, v in r["test"].items() if k not in ("name",)}} for r in records])

2026-07-06 19:05:14,011 INFO nb04: START all holdout schemes
2026-07-06 19:05:14,011 INFO nb04: START scheme 70_30
2026-07-06 19:05:34,729 INFO nb04: 70_30/decision_tree: fit on 1110831 rows in 20.7s
2026-07-06 19:05:36,715 INFO nb04: 70_30/decision_tree [test]: acc=0.7193 f1_macro=0.6624 roc_auc=0.9178 det=0.000ms


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   15.3s


2026-07-06 19:06:21,907 INFO nb04: 70_30/random_forest: fit on 1110831 rows in 45.2s


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   40.6s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.1s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.9s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.9s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.3s finished


2026-07-06 19:06:31,834 INFO nb04: 70_30/random_forest [test]: acc=0.7547 f1_macro=0.6775 roc_auc=0.9506 det=0.007ms
2026-07-06 19:06:46,005 INFO nb04: 70_30/svm: fit on 1110831 rows in 13.9s
2026-07-06 19:06:51,674 INFO nb04: 70_30/svm [test]: acc=0.4145 f1_macro=0.3859 roc_auc=0.8579 det=0.001ms
2026-07-06 19:06:58,963 INFO nb04: 70_30/naive_bayes: fit on 1110831 rows in 7.3s
2026-07-06 19:07:03,126 INFO nb04: 70_30/naive_bayes [test]: acc=0.4092 f1_macro=0.3428 roc_auc=0.8236 det=0.001ms
2026-07-06 19:07:08,670 INFO nb04: 70_30/knn: fit on 1110831 rows in 5.5s
2026-07-06 19:20:35,552 INFO nb04: 70_30/knn [test]: acc=0.7201 f1_macro=0.6292 roc_auc=0.927 det=0.539ms
2026-07-06 19:20:35,555 INFO nb04: END   scheme 70_30 (921.4s)
2026-07-06 19:20:35,556 INFO nb04: START scheme 80_20
2026-07-06 19:20:54,981 INFO nb04: 80_20/decision_tree: fit on 1269521 rows in 19.4s
2026-07-06 19:20:56,209 INFO nb04: 80_20/decision_tree [test]: acc=0.7124 f1_macro=0.6727 roc_auc=0.9183 det=0.000ms


...

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   36.7s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.7s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.7s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.7s finished


2026-07-06 19:36:47,543 INFO nb04: 70_20_10/random_forest [test]: acc=0.7543 f1_macro=0.6561 roc_auc=0.9506 det=0.006ms


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.3s finished


2026-07-06 19:36:52,722 INFO nb04: 70_20_10/random_forest [validation]: acc=0.7524 f1_macro=0.6982 roc_auc=0.9503 det=0.005ms
2026-07-06 19:37:02,754 INFO nb04: 70_20_10/svm: fit on 1110830 rows in 10.0s
2026-07-06 19:37:04,140 INFO nb04: 70_20_10/svm [test]: acc=0.4153 f1_macro=0.4059 roc_auc=0.8569 det=0.001ms
2026-07-06 19:37:06,638 INFO nb04: 70_20_10/svm [validation]: acc=0.4171 f1_macro=0.4045 roc_auc=0.8574 det=0.002ms
2026-07-06 19:37:11,777 INFO nb04: 70_20_10/naive_bayes: fit on 1110830 rows in 5.1s
2026-07-06 19:37:12,897 INFO nb04: 70_20_10/naive_bayes [test]: acc=0.4188 f1_macro=0.3460 roc_auc=0.8212 det=0.001ms
2026-07-06 19:37:14,985 INFO nb04: 70_20_10/naive_bayes [validation]: acc=0.4220 f1_macro=0.3468 roc_auc=0.8222 det=0.001ms
2026-07-06 19:37:19,516 INFO nb04: 70_20_10/knn: fit on 1110830 rows in 4.5s
2026-07-06 19:41:28,551 INFO nb04: 70_20_10/knn [test]: acc=0.7218 f1_macro=0.6241 roc_auc=0.9281 det=0.545ms
2026-07-06 19:49:38,686 INFO nb04: 70_20_10/knn [validat

,scheme,model,fit_s,accuracy,precision_macro,recall_macro,f1_macro,roc_auc,detection_time_ms,n_test
0,70_30,decision_tree,20.716,0.7193,0.6463,0.7681,0.6624,0.9178,0.0003,476071
1,70_30,random_forest,45.192,0.7547,0.6624,0.7904,0.6775,0.9506,0.0074,476071
2,70_30,svm,13.942,0.4145,0.3612,0.7554,0.3859,0.8579,0.0007,476071
3,70_30,naive_bayes,7.286,0.4092,0.3316,0.6062,0.3428,0.8236,0.0013,476071
4,70_30,knn,5.542,0.7201,0.6054,0.7901,0.6292,0.9270,0.5389,476071
5,80_20,decision_tree,19.423,0.7124,0.6550,0.7811,0.6727,0.9183,0.0003,317381
6,80_20,random_forest,41.585,0.7534,0.6604,0.7988,0.6773,0.9506,0.0053,317381
7,80_20,svm,9.840,0.4140,0.3686,0.7699,0.3950,0.8575,0.0012,317381
8,80_20,naive_bayes,4.173,0.4108,0.3470,0.6323,0.3569,0.8230,0.0007,317381
9,80_20,knn,3.867,0.7204,0.6073,0.8048,0.6334,0.9269,0.5075,317381


## 3. Five-fold stratified cross-validation (scheme `cv5`)

In [5]:
cv_dir = P.runs / "cv5"
cv_dir.mkdir(parents=True, exist_ok=True)
cv_scores_path = cv_dir / "cv_scores.csv"

if not RUN_CV:
    log.info("RUN_CV=False — skipping")
elif cv_scores_path.exists():
    log.info("SKIP cv5 — cv_scores.csv exists")
    cv_summary = pd.read_csv(cv_dir / "cv_summary.csv")
else:
    with nbc.timed(log, "5-fold stratified CV, all models"):
        scores, _ = cross_validate_models(X, y, top40, model_names=models_to_run,
                                          n_splits=5, target_per_class=TARGET_PER_CLASS)
    scores.to_csv(cv_scores_path, index=False)
    cv_summary = summarize_cv(scores)
    cv_summary.to_csv(cv_dir / "cv_summary.csv", index=False)
cv_summary if RUN_CV else None

2026-07-06 19:49:38,714 INFO nb04: START 5-fold stratified CV, all models
2026-07-06 19:49:57,819 INFO ids.evaluation.cross_val: CV decision_tree fold 1/5: f1_macro=0.6754 acc=0.7211 (18.0s)
2026-07-06 19:50:16,082 INFO ids.evaluation.cross_val: CV decision_tree fold 2/5: f1_macro=0.6639 acc=0.7243 (17.2s)
2026-07-06 19:50:34,106 INFO ids.evaluation.cross_val: CV decision_tree fold 3/5: f1_macro=0.6520 acc=0.7076 (17.3s)
2026-07-06 19:50:51,947 INFO ids.evaluation.cross_val: CV decision_tree fold 4/5: f1_macro=0.6743 acc=0.7480 (17.1s)
2026-07-06 19:51:09,796 INFO ids.evaluation.cross_val: CV decision_tree fold 5/5: f1_macro=0.6770 acc=0.7174 (17.1s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.3s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   34.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 19:51:50,904 INFO ids.evaluation.cross_val: CV random_forest fold 1/5: f1_macro=0.6420 acc=0.7453 (37.9s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   33.9s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.6s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 19:52:31,831 INFO ids.evaluation.cross_val: CV random_forest fold 2/5: f1_macro=0.6197 acc=0.7436 (37.3s)


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   12.9s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   34.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 19:53:54,064 INFO ids.evaluation.cross_val: CV random_forest fold 4/5: f1_macro=0.6529 acc=0.7441 (37.5s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   34.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 19:54:34,948 INFO ids.evaluation.cross_val: CV random_forest fold 5/5: f1_macro=0.6512 acc=0.7415 (37.5s)
2026-07-06 19:54:48,079 INFO ids.evaluation.cross_val: CV svm fold 1/5: f1_macro=0.3839 acc=0.4014 (11.2s)
2026-07-06 19:54:59,655 INFO ids.evaluation.cross_val: CV svm fold 2/5: f1_macro=0.3887 acc=0.4109 (9.9s)
2026-07-06 19:55:11,151 INFO ids.evaluation.cross_val: CV svm fold 3/5: f1_macro=0.3619 acc=0.4022 (9.7s)
2026-07-06 19:55:22,033 INFO ids.evaluation.cross_val: CV svm fold 4/5: f1_macro=0.3839 acc=0.4060 (9.2s)
2026-07-06 19:55:33,167 INFO ids.evaluation.cross_val: CV svm fold 5/5: f1_macro=0.3882 acc=0.4351 (9.3s)
2026-07-06 19:55:38,436 INFO ids.evaluation.cross_val: CV naive_bayes fold 1/5: f1_macro=0.3298 acc=0.2773 (3.6s)
2026-07-06 19:55:43,632 INFO ids.evaluation.cross_val: CV naive_bayes fold 2/5: f1_macro=0.3502 acc=0.4174 (3.7s)
2026-07-06 19:55:48,494 INFO ids.evaluation.cross_val: CV naive_bayes fold 3/5: f1_macro=0.3087 acc=0.1346 (3.4s)
2026-07-06

,model,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,roc_auc_mean,roc_auc_std
0,decision_tree,0.723678,0.014969,0.650937,0.008346,0.804992,0.032922,0.668532,0.010579,0.919143,0.003362
1,knn,0.719673,0.001633,0.601108,0.008376,0.815226,0.028540,0.622799,0.010700,0.926709,0.000590
2,naive_bayes,0.286340,0.102101,0.326760,0.013261,0.598274,0.027713,0.330652,0.015498,0.829227,0.001808
3,random_forest,0.743461,0.001437,0.630783,0.010142,0.808763,0.037654,0.640702,0.013313,0.950528,0.000254
4,svm,0.411116,0.013943,0.357154,0.013222,0.731961,0.025474,0.381326,0.011109,0.857660,0.011434


In [6]:
n_files = sum(1 for _ in P.runs.rglob("*") if _.is_file())
log.info("notebook 04 complete — %d artifact files under %s", n_files, P.runs)

2026-07-06 20:23:29,619 INFO nb04: notebook 04 complete — 142 artifact files under /Users/MAC/Projects/IDS/backend/artifacts/ft40_mi/cic2018/runs
